# AttnFool evaluation

Evaluates the seven classifiers used in the paper on a 1000-image subset of
ImageNet-2012 val, plus DETR on a 100-image subset of MS COCO 2017 val.

Classifiers: **ResNet50, ViT-T, ViT-B, ViT-B/384, DeiT-T, DeiT-B, DeiT-B/384**.

Before running, build the data subsets:

```bash
python -m datasets.prepare_data --imagenet-src /path/to/ImageNet/val
```

Knobs to tune cost: `NUM_IMAGENET`, `NUM_COCO`, `ATTACK_ITERS`, `BATCH_SIZE`,
`MODELS_TO_RUN`.

In [1]:
%pip install numpy
%pip install matplotlib
%pip install scipy
%pip install scikit-learn
%pip install torch
%pip install torchvision
%pip install torchaudio
%pip install transformers
%pip install datasets
%pip install wandb
%pip install timm
%pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.6 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 26.1.1
    Uninstalling pip-26.1.1:
      Successfully uninstalled pip-26.1.1
Note: you may need to restart the kernel to use updated p

In [2]:
import os, sys, math, time, json, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from PIL import Image

REPO = Path('.').resolve()
sys.path.insert(0, str(REPO))

from models.DeiT import (
    deit_tiny_patch16_224, deit_base_patch16_224, deit_base_patch16_384,
)
from models.vision_transformer import (
    vit_tiny_patch16_224, vit_base_patch16_224, vit_base_patch16_384,
)
from models.resnet import ResNet50
# NOTE: the Attention-Fool losses (l_kq / l_kq_star) and the PGD helpers
# (build_patch_mask, cosine_step_size, patch_token_index) are defined
# from-scratch in the "AttentionFool Implementation" section below and shadow
# any utils version, so they are intentionally NOT imported here.
from utils import (
    apply_patch, normalized_momentum, mu as IMNET_MU, std as IMNET_STD,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

device: cuda


## Config

In [3]:
NUM_IMAGENET = 1000          # images sampled by datasets.prepare_data
NUM_COCO     = 100

BATCH_SIZE   = 16            # drop if you OOM on the 384 models
ATTACK_ITERS = 250           # paper default; lower to e.g. 50 for a quick run
ATTACK_LR    = 8.0 / 255.0
ATTACK_MODE  = 'AttnFool_kq' # 'CE_loss' | 'AttnFool_kq' | 'AttnFool_kqstar'
ATTN_W       = 1.0
USE_MOMENTUM = False
PATCH_SIZE   = 16
PATCH_ROW    = 0
PATCH_COL    = 0
SEED         = 0

MODELS_TO_RUN = [
    'ResNet50',
    'ViT-T', 'ViT-B', 'ViT-B-384',
    'DeiT-T', 'DeiT-B', 'DeiT-B-384',
]

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

## Model registry

Each entry returns `(model, is_transformer, img_size)`. The transformer
flag controls whether we pull `(q,k)` from the model for the AttnFool
loss.

In [4]:
def _resnet50():
    m = ResNet50(pretrained=True)
    return m, False, 224

MODEL_FACTORY = {
    'ResNet50'  : _resnet50,
    'ViT-T'     : lambda: (vit_tiny_patch16_224(pretrained=True), True, 224),
    'ViT-B'     : lambda: (vit_base_patch16_224(pretrained=True), True, 224),
    'ViT-B-384' : lambda: (vit_base_patch16_384(pretrained=True), True, 384),
    'DeiT-T'    : lambda: (deit_tiny_patch16_224(pretrained=True), True, 224),
    'DeiT-B'    : lambda: (deit_base_patch16_224(pretrained=True), True, 224),
    'DeiT-B-384': lambda: (deit_base_patch16_384(pretrained=True), True, 384),
}

def forward(model, x, is_transformer):
    if is_transformer:
        out, qk_list = model(x)
        return out, qk_list
    return model(x), None

## ImageNet val loader (1000 images)

In [5]:
IMAGENET_DIR = REPO / 'data' / 'imagenet_val'
assert IMAGENET_DIR.is_dir(), (
    f'Missing {IMAGENET_DIR}. Run `python -m datasets.prepare_data --imagenet-src <path>` first.'
)

def make_imagenet_loader(img_size, batch_size=BATCH_SIZE):
    resize = int(img_size / 0.875) if img_size == 224 else img_size
    tfm = transforms.Compose([
        transforms.Resize(resize, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMNET_MU, std=IMNET_STD),
    ])
    ds = torchvision.datasets.ImageFolder(str(IMAGENET_DIR), transform=tfm)
    print(f'  imagenet({img_size}): {len(ds)} images, {len(ds.classes)} classes')
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False,
                                       num_workers=2, pin_memory=True)

## Attack matrix

Six attack configurations, each split into three groups so each group can be re-run independently (matching the original training split). Group 1 includes ResNet50 for Lce-only attacks; the four attention attacks (Lkq / Lkq\*) skip ResNet50 since it has no attention.

In [6]:
# Per-attack runner. Spawns one subprocess per model (so CUDA memory is fully
# released on exit) with the requested attack mode and momentum flag, and
# stores results under results[tag][name] so the six attack configurations
# coexist in results.json.
import subprocess, os
from pathlib import Path

RESULTS_PATH = Path('results.json')
results = json.loads(RESULTS_PATH.read_text()) if RESULTS_PATH.exists() else {}

# Group layout mirrors the original training split: G1 covers ResNet+small ViTs,
# G2 covers DeiTs at 224, G3 covers the heavy 384 models. Attention attacks
# skip ResNet50 since it has no attention.
G1_ALL = ['ResNet50', 'ViT-T', 'ViT-B']
G1_TF  = ['ViT-T', 'ViT-B']
G2     = ['DeiT-T', 'DeiT-B']
G3     = ['ViT-B-384', 'DeiT-B-384']

def run_attack(tag, attack_mode, use_momentum, models):
    bucket = results.setdefault(tag, {})
    for name in models:
        print(f'\n=== [{tag}] {name} ===', flush=True)
        cmd = [
            sys.executable, '-u', 'run_model.py',
            '--name', name,
            '--num-imagenet', str(NUM_IMAGENET),
            '--batch-size', str(BATCH_SIZE),
            '--attack-iters', str(ATTACK_ITERS),
            '--attack-lr', str(ATTACK_LR),
            '--attack-mode', attack_mode,
            '--attn-w', str(ATTN_W),
            '--patch-size', str(PATCH_SIZE),
            '--patch-row', str(PATCH_ROW),
            '--patch-col', str(PATCH_COL),
            '--seed', str(SEED),
        ]
        if use_momentum:
            cmd.append('--use-momentum')
        env = {**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, env=env, cwd=str(REPO), bufsize=1)
        last_json = None
        for line in proc.stdout:
            line = line.rstrip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                if isinstance(obj, dict) and 'clean_acc' in obj:
                    last_json = obj
                    continue
            except json.JSONDecodeError:
                pass
            print(line, flush=True)
        proc.wait()
        if proc.returncode != 0 or last_json is None:
            print(f'  [{tag}/{name}] FAILED (exit {proc.returncode})', flush=True)
            continue
        bucket[name] = {'clean_acc': last_json['clean_acc'],
                        'attnfool_acc': last_json['attnfool_acc']}
        RESULTS_PATH.write_text(json.dumps(results, indent=2))
    return bucket


### 1. PGD + Lce

#### Group 1: ResNet50, ViT-T, ViT-B

In [7]:
run_attack('pgd_lce', 'pgd_lce', use_momentum=False, models=G1_ALL)
print(json.dumps(results['pgd_lce'], indent=2))


=== [pgd_lce] ResNet50 ===
[ResNet50] device=cuda
[ResNet50] imagenet(224): 1000 images
[ResNet50] clean acc : 0.7960  [2.6s]
[ResNet50] attnfool  : 0.6350  [851.4s]

=== [pgd_lce] ViT-T ===
[ViT-T] device=cuda
[ViT-T] imagenet(224): 1000 images
[ViT-T] clean acc : 0.7350  [3.0s]
[ViT-T] attnfool  : 0.1250  [391.4s]

=== [pgd_lce] ViT-B ===
[ViT-B] device=cuda
[ViT-B] imagenet(224): 1000 images
[ViT-B] clean acc : 0.8380  [7.3s]
[ViT-B] attnfool  : 0.5050  [3233.1s]
{
  "ResNet50": {
    "clean_acc": 0.796,
    "attnfool_acc": 0.635
  },
  "ViT-T": {
    "clean_acc": 0.735,
    "attnfool_acc": 0.125
  },
  "ViT-B": {
    "clean_acc": 0.838,
    "attnfool_acc": 0.505
  },
  "DeiT-T": {
    "clean_acc": 0.71,
    "attnfool_acc": 0.203
  },
  "DeiT-B": {
    "clean_acc": 0.816,
    "attnfool_acc": 0.38
  },
  "ViT-B-384": {
    "clean_acc": 0.838,
    "attnfool_acc": 0.544
  }
}


#### Group 2: DeiT-T, DeiT-B

In [8]:
run_attack('pgd_lce', 'pgd_lce', use_momentum=False, models=G2)
print(json.dumps(results['pgd_lce'], indent=2))


=== [pgd_lce] DeiT-T ===
[DeiT-T] device=cuda
[DeiT-T] imagenet(224): 1000 images
[DeiT-T] clean acc : 0.7100  [2.8s]
[DeiT-T] attnfool  : 0.2030  [391.1s]

=== [pgd_lce] DeiT-B ===
[DeiT-B] device=cuda
[DeiT-B] imagenet(224): 1000 images
[DeiT-B] clean acc : 0.8160  [7.3s]
[DeiT-B] attnfool  : 0.3800  [3244.4s]
{
  "ResNet50": {
    "clean_acc": 0.796,
    "attnfool_acc": 0.635
  },
  "ViT-T": {
    "clean_acc": 0.735,
    "attnfool_acc": 0.125
  },
  "ViT-B": {
    "clean_acc": 0.838,
    "attnfool_acc": 0.505
  },
  "DeiT-T": {
    "clean_acc": 0.71,
    "attnfool_acc": 0.203
  },
  "DeiT-B": {
    "clean_acc": 0.816,
    "attnfool_acc": 0.38
  },
  "ViT-B-384": {
    "clean_acc": 0.838,
    "attnfool_acc": 0.544
  }
}


#### Group 3: ViT-B-384, DeiT-B-384

In [9]:
run_attack('pgd_lce', 'pgd_lce', use_momentum=False, models=G3)
print(json.dumps(results['pgd_lce'], indent=2))


=== [pgd_lce] ViT-B-384 ===
[ViT-B-384] device=cuda
[ViT-B-384] imagenet(384): 1000 images
[ViT-B-384] clean acc : 0.8380  [21.2s]


KeyboardInterrupt: 

### 2. PGD + Lce + Lkq

#### Group 1: ViT-T, ViT-B

In [10]:
run_attack('pgd_lce_lkq', 'AttnFool_kq', use_momentum=False, models=G1_TF)
print(json.dumps(results['pgd_lce_lkq'], indent=2))


=== [pgd_lce_lkq] ViT-T ===
[ViT-T] device=cuda
[ViT-T] imagenet(224): 1000 images
[ViT-T] clean acc : 0.7350  [7.7s]
[ViT-T] attnfool  : 0.1630  [483.5s]

=== [pgd_lce_lkq] ViT-B ===
[ViT-B] device=cuda
[ViT-B] imagenet(224): 1000 images
[ViT-B] clean acc : 0.8380  [6.9s]
[ViT-B] attnfool  : 0.4600  [3745.4s]
{
  "ViT-T": {
    "clean_acc": 0.735,
    "attnfool_acc": 0.163
  },
  "ViT-B": {
    "clean_acc": 0.838,
    "attnfool_acc": 0.46
  }
}


#### Group 2: DeiT-T, DeiT-B

In [11]:
run_attack('pgd_lce_lkq', 'AttnFool_kq', use_momentum=False, models=G2)
print(json.dumps(results['pgd_lce_lkq'], indent=2))


=== [pgd_lce_lkq] DeiT-T ===
[DeiT-T] device=cuda
[DeiT-T] imagenet(224): 1000 images
[DeiT-T] clean acc : 0.7100  [2.1s]
[DeiT-T] attnfool  : 0.1850  [477.8s]

=== [pgd_lce_lkq] DeiT-B ===
[DeiT-B] device=cuda
[DeiT-B] imagenet(224): 1000 images
[DeiT-B] clean acc : 0.8160  [6.9s]
[DeiT-B] attnfool  : 0.3520  [3727.4s]
{
  "ViT-T": {
    "clean_acc": 0.735,
    "attnfool_acc": 0.163
  },
  "ViT-B": {
    "clean_acc": 0.838,
    "attnfool_acc": 0.46
  },
  "DeiT-T": {
    "clean_acc": 0.71,
    "attnfool_acc": 0.185
  },
  "DeiT-B": {
    "clean_acc": 0.816,
    "attnfool_acc": 0.352
  }
}


#### Group 3: ViT-B-384, DeiT-B-384

In [ ]:
run_attack('pgd_lce_lkq', 'AttnFool_kq', use_momentum=False, models=G3)
print(json.dumps(results['pgd_lce_lkq'], indent=2))

### 3. PGD + Lce + Lkq*

#### Group 1: ViT-T, ViT-B

In [12]:
run_attack('pgd_lce_lkqstar', 'AttnFool_kqstar', use_momentum=False, models=G1_TF)
print(json.dumps(results['pgd_lce_lkqstar'], indent=2))


=== [pgd_lce_lkqstar] ViT-T ===
[ViT-T] device=cuda
[ViT-T] imagenet(224): 1000 images
[ViT-T] clean acc : 0.7350  [2.0s]
[ViT-T] attnfool  : 0.1210  [479.5s]

=== [pgd_lce_lkqstar] ViT-B ===
[ViT-B] device=cuda
[ViT-B] imagenet(224): 1000 images
[ViT-B] clean acc : 0.8380  [6.9s]
[ViT-B] attnfool  : 0.4830  [3758.2s]
{
  "ViT-T": {
    "clean_acc": 0.735,
    "attnfool_acc": 0.121
  },
  "ViT-B": {
    "clean_acc": 0.838,
    "attnfool_acc": 0.483
  }
}


#### Group 2: DeiT-T, DeiT-B

In [13]:
run_attack('pgd_lce_lkqstar', 'AttnFool_kqstar', use_momentum=False, models=G2)
print(json.dumps(results['pgd_lce_lkqstar'], indent=2))


=== [pgd_lce_lkqstar] DeiT-T ===
[DeiT-T] device=cuda
[DeiT-T] imagenet(224): 1000 images
[DeiT-T] clean acc : 0.7100  [2.1s]
[DeiT-T] attnfool  : 0.1820  [476.0s]

=== [pgd_lce_lkqstar] DeiT-B ===
[DeiT-B] device=cuda
[DeiT-B] imagenet(224): 1000 images
[DeiT-B] clean acc : 0.8160  [6.9s]
[DeiT-B] attnfool  : 0.3510  [3693.4s]
{
  "ViT-T": {
    "clean_acc": 0.735,
    "attnfool_acc": 0.121
  },
  "ViT-B": {
    "clean_acc": 0.838,
    "attnfool_acc": 0.483
  },
  "DeiT-T": {
    "clean_acc": 0.71,
    "attnfool_acc": 0.182
  },
  "DeiT-B": {
    "clean_acc": 0.816,
    "attnfool_acc": 0.351
  }
}


#### Group 3: ViT-B-384, DeiT-B-384

In [ ]:
run_attack('pgd_lce_lkqstar', 'AttnFool_kqstar', use_momentum=False, models=G3)
print(json.dumps(results['pgd_lce_lkqstar'], indent=2))

### 4. PGD + Lce + momentum

#### Group 1: ResNet50, ViT-T, ViT-B

In [14]:
run_attack('pgd_lce_mom', 'pgd_lce', use_momentum=True, models=G1_ALL)
print(json.dumps(results['pgd_lce_mom'], indent=2))


=== [pgd_lce_mom] ResNet50 ===
[ResNet50] device=cuda
[ResNet50] imagenet(224): 1000 images
[ResNet50] clean acc : 0.7960  [2.4s]
[ResNet50] attnfool  : 0.5930  [822.8s]

=== [pgd_lce_mom] ViT-T ===
[ViT-T] device=cuda
[ViT-T] imagenet(224): 1000 images
[ViT-T] clean acc : 0.7350  [3.3s]
[ViT-T] attnfool  : 0.0230  [398.1s]

=== [pgd_lce_mom] ViT-B ===
[ViT-B] device=cuda
[ViT-B] imagenet(224): 1000 images
[ViT-B] clean acc : 0.8380  [7.0s]
[ViT-B] attnfool  : 0.3810  [3246.6s]
{
  "ResNet50": {
    "clean_acc": 0.796,
    "attnfool_acc": 0.593
  },
  "ViT-T": {
    "clean_acc": 0.735,
    "attnfool_acc": 0.023
  },
  "ViT-B": {
    "clean_acc": 0.838,
    "attnfool_acc": 0.381
  }
}


#### Group 2: DeiT-T, DeiT-B

In [15]:
run_attack('pgd_lce_mom', 'pgd_lce', use_momentum=True, models=G2)
print(json.dumps(results['pgd_lce_mom'], indent=2))


=== [pgd_lce_mom] DeiT-T ===
[DeiT-T] device=cuda
[DeiT-T] imagenet(224): 1000 images
[DeiT-T] clean acc : 0.7100  [2.0s]
[DeiT-T] attnfool  : 0.0100  [392.3s]

=== [pgd_lce_mom] DeiT-B ===
[DeiT-B] device=cuda
[DeiT-B] imagenet(224): 1000 images
[DeiT-B] clean acc : 0.8160  [7.3s]
[DeiT-B] attnfool  : 0.1660  [3235.0s]
{
  "ResNet50": {
    "clean_acc": 0.796,
    "attnfool_acc": 0.593
  },
  "ViT-T": {
    "clean_acc": 0.735,
    "attnfool_acc": 0.023
  },
  "ViT-B": {
    "clean_acc": 0.838,
    "attnfool_acc": 0.381
  },
  "DeiT-T": {
    "clean_acc": 0.71,
    "attnfool_acc": 0.01
  },
  "DeiT-B": {
    "clean_acc": 0.816,
    "attnfool_acc": 0.166
  }
}


#### Group 3: ViT-B-384, DeiT-B-384

In [ ]:
run_attack('pgd_lce_mom', 'pgd_lce', use_momentum=True, models=G3)
print(json.dumps(results['pgd_lce_mom'], indent=2))

### 5. PGD + Lce + momentum + Lkq

#### Group 1: ViT-T, ViT-B

In [16]:
run_attack('pgd_lce_mom_lkq', 'AttnFool_kq', use_momentum=True, models=G1_TF)
print(json.dumps(results['pgd_lce_mom_lkq'], indent=2))


=== [pgd_lce_mom_lkq] ViT-T ===
[ViT-T] device=cuda
[ViT-T] imagenet(224): 1000 images
[ViT-T] clean acc : 0.7350  [2.0s]
[ViT-T] attnfool  : 0.0430  [481.3s]

=== [pgd_lce_mom_lkq] ViT-B ===
[ViT-B] device=cuda
[ViT-B] imagenet(224): 1000 images
[ViT-B] clean acc : 0.8380  [7.3s]


KeyboardInterrupt: 

#### Group 2: DeiT-T, DeiT-B

In [ ]:
run_attack('pgd_lce_mom_lkq', 'AttnFool_kq', use_momentum=True, models=G2)
print(json.dumps(results['pgd_lce_mom_lkq'], indent=2))

#### Group 3: ViT-B-384, DeiT-B-384

In [ ]:
run_attack('pgd_lce_mom_lkq', 'AttnFool_kq', use_momentum=True, models=G3)
print(json.dumps(results['pgd_lce_mom_lkq'], indent=2))

### 6. PGD + Lce + momentum + Lkq*

#### Group 1: ViT-T, ViT-B

In [ ]:
run_attack('pgd_lce_mom_lkqstar', 'AttnFool_kqstar', use_momentum=True, models=G1_TF)
print(json.dumps(results['pgd_lce_mom_lkqstar'], indent=2))

#### Group 2: DeiT-T, DeiT-B

In [ ]:
run_attack('pgd_lce_mom_lkqstar', 'AttnFool_kqstar', use_momentum=True, models=G2)
print(json.dumps(results['pgd_lce_mom_lkqstar'], indent=2))

#### Group 3: ViT-B-384, DeiT-B-384

In [ ]:
run_attack('pgd_lce_mom_lkqstar', 'AttnFool_kqstar', use_momentum=True, models=G3)
print(json.dumps(results['pgd_lce_mom_lkqstar'], indent=2))

## Results summary

In [ ]:
# Summary table across all six attack configurations.
# Clean acc is reported by run_model.py alongside each attack run; we surface
# the first available clean acc per model (it does not depend on the attack).
ATTACK_TAGS = [
    ('pgd_lce',             'PGD+Lce'),
    ('pgd_lce_lkq',         'PGD+Lce+Lkq'),
    ('pgd_lce_lkqstar',     'PGD+Lce+Lkq*'),
    ('pgd_lce_mom',         'PGD+Lce+Mom'),
    ('pgd_lce_mom_lkq',     'PGD+Lce+Mom+Lkq'),
    ('pgd_lce_mom_lkqstar', 'PGD+Lce+Mom+Lkq*'),
]
MODEL_ORDER = ['ResNet50', 'ViT-T', 'ViT-B', 'ViT-B-384', 'DeiT-T', 'DeiT-B', 'DeiT-B-384']

def _clean_for(model):
    for tag, _ in ATTACK_TAGS:
        entry = results.get(tag, {}).get(model)
        if entry is not None:
            return entry['clean_acc']
    return None

def _fmt(v):
    return f'{v*100:6.2f}%' if isinstance(v, (int, float)) else '   -   '

header = ['Model', 'Clean'] + [label for _, label in ATTACK_TAGS]
rows = []
for model in MODEL_ORDER:
    row = [model, _fmt(_clean_for(model))]
    for tag, _ in ATTACK_TAGS:
        entry = results.get(tag, {}).get(model)
        row.append(_fmt(entry['attnfool_acc']) if entry else '   -   ')
    rows.append(row)

widths = [max(len(str(r[i])) for r in [header] + rows) for i in range(len(header))]
def _line(cells):
    return ' | '.join(str(c).ljust(widths[i]) for i, c in enumerate(cells))

print(_line(header))
print('-+-'.join('-' * w for w in widths))
for r in rows:
    print(_line(r))

# Also emit a markdown table so it renders nicely in the notebook.
from IPython.display import Markdown, display
md_lines = ['| ' + ' | '.join(header) + ' |',
            '| ' + ' | '.join('---' for _ in header) + ' |']
for r in rows:
    md_lines.append('| ' + ' | '.join(str(c) for c in r) + ' |')
display(Markdown('\n'.join(md_lines)))
